# 迁移验证
验证 ZettaPark 迁移后的数据质量和业务逻辑正确性。

| 检查类别 | 内容 |
|---------|------|
| 行数合理性 | 各层表有数据，行数符合预期 |
| Bronze → Silver 一致性 | Silver 不丢行，销售明细完整 |
| 关键列无空值 | 主键、外键均不为 NULL |
| 数据清洗效果 | gender / marital_status / country 标准化正确 |
| 维度完整性 | surrogate key 无重复 |
| 外键完整性 | fact_sales 所有外键均能关联到 dim 表 |
| 业务指标合理性 | 数量 > 0，country 覆盖率 > 50% |

## Setup Connection

In [1]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
S = os.environ["CLICKZETTA_SCHEMA"]
print(f"Connected. Schema: {S}")

Connected. Schema: public


## Check Helper

In [2]:
results = []

def check(name, sql, expect_zero=True, desc="", warn_only=False):
    rows = session.sql(sql).collect()
    val = rows[0][0] if rows else 0
    passed = (val == 0) if expect_zero else (val > 0)
    if passed:       status = "✓"
    elif warn_only:  status = "⚠"
    else:            status = "✗"
    results.append((status, name, val, desc))
    print(f"  {status}  {name:<55} {val}")

## 1. 行数合理性

In [3]:
print("── 1. 行数合理性 ──")
check("Bronze crm_cust_info 有数据",
      f"SELECT COUNT(*) FROM {S}.crm_cust_info", expect_zero=False)
check("Bronze crm_sales_details 有数据",
      f"SELECT COUNT(*) FROM {S}.crm_sales_details", expect_zero=False)
check("Gold fact_sales 有数据",
      f"SELECT COUNT(*) FROM {S}.fact_sales", expect_zero=False)

── 1. 行数合理性 ──


  ✓  Bronze crm_cust_info 有数据                                18494
  ✓  Bronze crm_sales_details 有数据                            60398
  ✓  Gold fact_sales 有数据                                     89833


## 2. Bronze → Silver 行数一致性

In [4]:
print("── 2. Bronze → Silver 行数一致性 ──")
check("Silver crm_customers 行数 ≤ Bronze crm_cust_info",
      f"""SELECT CASE WHEN
          (SELECT COUNT(*) FROM {S}.crm_customers) >
          (SELECT COUNT(*) FROM {S}.crm_cust_info)
      THEN 1 ELSE 0 END""")
check("Silver crm_sales 行数 = Bronze crm_sales_details",
      f"""SELECT ABS(
          (SELECT COUNT(*) FROM {S}.crm_sales) -
          (SELECT COUNT(*) FROM {S}.crm_sales_details))""")

── 2. Bronze → Silver 行数一致性 ──


  ✓  Silver crm_customers 行数 ≤ Bronze crm_cust_info          0
  ✓  Silver crm_sales 行数 = Bronze crm_sales_details          0


## 3. 关键列无空值

In [5]:
print("── 3. 关键列无空值 ──")
check("Silver crm_customers.customer_id 无空值",
      f"SELECT COUNT(*) FROM {S}.crm_customers WHERE customer_id IS NULL")
check("Silver crm_customers.customer_number 无空值",
      f"SELECT COUNT(*) FROM {S}.crm_customers WHERE customer_number IS NULL")
check("Silver crm_sales.order_number 无空值",
      f"SELECT COUNT(*) FROM {S}.crm_sales WHERE order_number IS NULL")
check("Gold dim_customers.customer_key 无空值",
      f"SELECT COUNT(*) FROM {S}.dim_customers WHERE customer_key IS NULL")
check("Gold dim_products.product_key 无空值",
      f"SELECT COUNT(*) FROM {S}.dim_products WHERE product_key IS NULL")

── 3. 关键列无空值 ──


  ✓  Silver crm_customers.customer_id 无空值                    0


  ✓  Silver crm_customers.customer_number 无空值                0


  ✓  Silver crm_sales.order_number 无空值                       0


  ✓  Gold dim_customers.customer_key 无空值                     0


  ✓  Gold dim_products.product_key 无空值                       0


## 4. Silver 数据清洗效果

In [6]:
print("── 4. Silver 数据清洗效果 ──")
check("crm_customers.gender 只含标准值",
      f"SELECT COUNT(*) FROM {S}.crm_customers WHERE gender NOT IN ('Male','Female','n/a')")
check("crm_customers.marital_status 只含标准值",
      f"SELECT COUNT(*) FROM {S}.crm_customers WHERE marital_status NOT IN ('Single','Married','n/a')")
check("erp_customers.gender 只含标准值",
      f"SELECT COUNT(*) FROM {S}.erp_customers WHERE gender NOT IN ('Male','Female','n/a')")
check("erp_customer_location.country 无空字符串",
      f"SELECT COUNT(*) FROM {S}.erp_customer_location WHERE country = ''")

── 4. Silver 数据清洗效果 ──


  ✓  crm_customers.gender 只含标准值                              0


  ✓  crm_customers.marital_status 只含标准值                      0


  ✓  erp_customers.gender 只含标准值                              0
  ✓  erp_customer_location.country 无空字符串                     0


## 5. Gold 维度完整性

In [7]:
print("── 5. Gold 维度完整性 ──")
check("dim_customers.customer_key 无重复",
      f"""SELECT COUNT(*) FROM (
          SELECT customer_key, COUNT(*) n FROM {S}.dim_customers
          GROUP BY customer_key HAVING n > 1)""")
check("dim_products.product_key 无重复",
      f"""SELECT COUNT(*) FROM (
          SELECT product_key, COUNT(*) n FROM {S}.dim_products
          GROUP BY product_key HAVING n > 1)""")
check("dim_customers.customer_id 无重复",
      f"""SELECT COUNT(*) FROM (
          SELECT customer_id, COUNT(*) n FROM {S}.dim_customers
          GROUP BY customer_id HAVING n > 1)""",
      warn_only=True, desc="源数据 crm_cust_info 本身存在重复 customer_id")

── 5. Gold 维度完整性 ──


  ✓  dim_customers.customer_key 无重复                          0


  ✓  dim_products.product_key 无重复                            0


  ⚠  dim_customers.customer_id 无重复                           5


## 6. fact_sales 外键关联完整性

In [8]:
print("── 6. fact_sales 外键关联完整性 ──")
check("fact_sales.product_key 均存在于 dim_products",
      f"""SELECT COUNT(*) FROM {S}.fact_sales f
          LEFT JOIN {S}.dim_products p ON f.product_key = p.product_key
          WHERE f.product_key IS NOT NULL AND p.product_key IS NULL""")
check("fact_sales.customer_key 均存在于 dim_customers",
      f"""SELECT COUNT(*) FROM {S}.fact_sales f
          LEFT JOIN {S}.dim_customers c ON f.customer_key = c.customer_key
          WHERE f.customer_key IS NOT NULL AND c.customer_key IS NULL""")

── 6. fact_sales 外键关联完整性 ──


  ✓  fact_sales.product_key 均存在于 dim_products                0


  ✓  fact_sales.customer_key 均存在于 dim_customers              0


## 7. 业务指标合理性

In [9]:
print("── 7. 业务指标合理性 ──")
check("crm_sales.sales_amount 无负值",
      f"SELECT COUNT(*) FROM {S}.crm_sales WHERE sales_amount < 0",
      warn_only=True, desc="源数据存在负值（退货/冲销单），非迁移问题")
check("crm_sales.quantity 无零或负值",
      f"SELECT COUNT(*) FROM {S}.crm_sales WHERE quantity <= 0")
check("dim_customers 有 country 信息的比例 > 50%",
      f"""SELECT CASE WHEN
          (SELECT COUNT(*) FROM {S}.dim_customers WHERE country IS NOT NULL AND country <> 'n/a') * 100.0 /
          NULLIF((SELECT COUNT(*) FROM {S}.dim_customers), 0) < 50
      THEN 1 ELSE 0 END""")

── 7. 业务指标合理性 ──


  ⚠  crm_sales.sales_amount 无负值                              3


  ✓  crm_sales.quantity 无零或负值                                0
  ✓  dim_customers 有 country 信息的比例 > 50%                     0


## 汇总结果

In [10]:
passed = sum(1 for r in results if r[0] == "✓")
warned = sum(1 for r in results if r[0] == "⚠")
failed = sum(1 for r in results if r[0] == "✗")
total  = len(results)

print(f"\n{'='*60}")
print(f"验证结果: {passed}/{total} 通过", end="")
if warned: print(f"  {warned} 项源数据质量警告", end="")
if failed:
    print(f"  {failed} 项失败")
    for s, name, val, desc in results:
        if s == "✗": print(f"  ✗  {name}  (值={val}  {desc})")
else:
    print("  — 迁移验证全部通过")
if warned:
    print("\n源数据质量警告（非迁移问题）：")
    for s, name, val, desc in results:
        if s == "⚠": print(f"  ⚠  {name}  (值={val})  {desc}")
print('='*60)


验证结果: 20/22 通过  2 项源数据质量警告  — 迁移验证全部通过

源数据质量警告（非迁移问题）：
  ⚠  dim_customers.customer_id 无重复  (值=5)  源数据 crm_cust_info 本身存在重复 customer_id
  ⚠  crm_sales.sales_amount 无负值  (值=3)  源数据存在负值（退货/冲销单），非迁移问题
